# 🔍 02. Unsupervised Anomaly Detection via Deep Autoencoder
### Project: XAI System in Banking: AI Anomaly-Driven Credit Card Fraud Detection and Interpretation
**Objective:** Train an unsupervised Deep Autoencoder strictly on legitimate transactions ($y=0$) to model the normal banking behavior manifold. Transactions with high reconstruction errors are flagged as anomalies.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.data.preprocessing import DataPipeline
from src.models.autoencoder import DeepAutoencoder
from src.training.autoencoder_trainer import AutoencoderTrainer
from src.utils.device import get_device

device = get_device("auto")
print(f"Execution Device: {device}")

## 1. Load Processed Dataset
We load the preprocessed splits, noting that $X_{\text{train\_normal}}$ contains only class 0 transactions.

In [ ]:
pipeline = DataPipeline(processed_dir=str(ROOT_DIR / "data/processed"))
data = pipeline.load_processed()
input_dim = data.X_train.shape[1]
print(f"Features: {input_dim}")
print(f"Normal training samples: {len(data.X_train_normal):,}")
print(f"Validation samples: {len(data.X_val):,} (Frauds: {np.sum(data.y_val == 1)})")
print(f"Test samples: {len(data.X_test):,} (Frauds: {np.sum(data.y_test == 1)})")

## 2. Load Trained Deep Autoencoder
The Autoencoder compresses the 33-dimensional feature space into an 8-dimensional latent bottleneck and reconstructs the original vector.

In [ ]:
model_path = ROOT_DIR / "results/models/autoencoder.pt"
autoencoder = DeepAutoencoder(input_dim=input_dim, latent_dim=8).to(device)

if model_path.exists():
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    autoencoder.load_state_dict(state)
    threshold = ckpt.get("threshold", 0.015)
    print(f"Loaded trained Autoencoder from {model_path}")
    print(f"Calibrated 98th Percentile Anomaly Threshold: {threshold:.6f}")
else:
    print("Pretrained checkpoint not found; please execute python scripts/train.py first.")

## 3. Reconstruction Error Separation Analysis
We evaluate reconstruction errors across normal vs fraudulent transactions in the test set.

In [ ]:
autoencoder.eval()
test_tensor = torch.as_tensor(data.X_test, dtype=torch.float32, device=device)
with torch.no_grad():
    recon_errors = autoencoder.compute_reconstruction_error(test_tensor, reduction="mean").cpu().numpy()

normal_errors = recon_errors[data.y_test == 0]
fraud_errors = recon_errors[data.y_test == 1]

print(f"Mean Normal Reconstruction MSE: {normal_errors.mean():.6f} (Median: {np.median(normal_errors):.6f})")
print(f"Mean Fraud Reconstruction MSE:  {fraud_errors.mean():.6f} (Median: {np.median(fraud_errors):.6f})")
print(f"Separation Ratio: {fraud_errors.mean() / normal_errors.mean():.2f}x higher error on fraud!")

plt.figure(figsize=(10, 5))
sns.kdeplot(np.log1p(normal_errors), label="Normal Transactions (Class 0)", color="#2b5c8f", lw=2)
sns.kdeplot(np.log1p(fraud_errors), label="Fraudulent Transactions (Class 1)", color="#d9534f", lw=2)
if 'threshold' in locals():
    plt.axvline(np.log1p(threshold), color="black", linestyle="--", label=f"Anomaly Threshold ({threshold:.4f})")
plt.title("Reconstruction Error Distribution (Log-Scale MSE)", fontsize=12, fontweight="bold")
plt.xlabel("Log(1 + Reconstruction MSE)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Latent Space Representation (Bottleneck Projection)
We extract the 8-dimensional bottleneck latents $z$ and project them to 2D using PCA to visualize manifold clustering.

In [ ]:
from sklearn.decomposition import PCA

with torch.no_grad():
    _, z_test = autoencoder(test_tensor[:2000])
    z_np = z_test.cpu().numpy()
    y_sub = data.y_test[:2000]

pca = PCA(n_components=2)
z_2d = pca.fit_transform(z_np)

plt.figure(figsize=(8, 6))
plt.scatter(z_2d[y_sub == 0, 0], z_2d[y_sub == 0, 1], c="#2b5c8f", alpha=0.3, s=15, label="Legitimate")
plt.scatter(z_2d[y_sub == 1, 0], z_2d[y_sub == 1, 1], c="#d9534f", alpha=0.9, s=40, edgecolors="k", label="Fraud")
plt.title("Autoencoder Latent Manifold Projection (First 2 PCA Components)", fontsize=12, fontweight="bold")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.legend()
plt.tight_layout()
plt.show()